# 2.as_retriever检索器示例

知识点讲解：as_retriever() 把向量数据库转换为检索器
1. VectorStore 与 Retriever 的本质区别
   - VectorStore 是"存储层"，负责增删改查向量，方法名各异（similarity_search、
     max_marginal_relevance_search 等），不同实现签名可能不一致
   - Retriever 是"接口层"，统一暴露 invoke(query) -> list[Document]，并实现 Runnable 协议，
     可直接参与 LCEL 管道编排
   - as_retriever() 就是把存储层包装为标准 Runnable 的适配器（VectorStoreRetriever）

2. as_retriever() 的三种 search_type
   - "similarity"（默认）：纯相似度排序，取 Top-K，不做阈值过滤
   - "similarity_score_threshold"：相似度 + 阈值过滤，底层调用
     similarity_search_with_relevance_scores，search_kwargs 必须包含 score_threshold
   - "mmr"：最大边际相关性，在相关性与多样性之间权衡，避免返回一堆重复内容

3. search_kwargs 常用参数与强制约束
   - k：返回文档条数
   - score_threshold：相关性阈值（仅 similarity_score_threshold 生效）
   - fetch_k：MMR 候选池大小（仅 mmr 生效，默认 20）
   - lambda_mult：MMR 相关性/多样性权重（仅 mmr 生效，默认 0.5）
   - filter：元数据过滤条件，不同向量库语法不同
   - 强制约束：search_type 配置为 similarity_score_threshold 后，search_kwargs 必须
     添加 score_threshold，否则会直接报错；各参数确切含义要结合对应底层函数一起看

4. 检索模式全景与选择
   - similarity（默认） → similarity_search：通用，约 80% 场合够用
   - similarity_score_threshold → similarity_search_with_relevance_scores：小库/要求准确、
     必须排除噪声时使用，必须传 score_threshold
   - mmr → max_marginal_relevance_search：追求创新/创意/多样性、"有哪些"类列举式提问
   - 经验结论：约 80% 场合用相似性搜索即可得到不错效果；只有追求多样性时才考虑 MMR
   - 使用相似性搜索时，尽可能选 similarity_search_with_relevance_scores 并传阈值，
     确保向量库数据较少时也不把不相关数据检索出来

5. 为什么 Retriever 是 RAG 的标准接口
   - 因为实现了 Runnable，可以写成 {"context": retriever, "question": ...} | prompt | llm
   - 换底层存储（FAISS → Weaviate → Pinecone）时，上层链代码完全不用改
   - 天然支持回调追踪（callbacks）、异步（ainvoke）、批量（batch）、运行时配置
   - 它的真正价值在于把检索器变成 Runnable，从而让查询转换阶段的多种策略
     （多查询重写、RAG-Fusion、问题分解、Step-Back、HyDE、集成检索）
     都能以"包装一个基础 retriever"的方式实现
   - 可作为优化对象的 RAG 组件共 5 处：query、TextSplitter、VectorStore、Retriever、Prompt，
     本节同时涉及 TextSplitter、VectorStore、Retriever 三处

6. WeaviateVectorStore 关键参数
   - client：weaviate 连接客户端，通过 connect_to_wcs 连接云端集群
   - index_name：Weaviate 中的 Collection 名称（相当于表名），不存在会自动创建
   - text_key：正文字段名，Weaviate 用该字段存储 Document.page_content
   - embedding：嵌入模型，入库和检索都会用它

7. 本例完整数据流水线
   - Markdown 文件 → UnstructuredMarkdownLoader.load() → 原始 Document
   - → RecursiveCharacterTextSplitter.split_documents() → 若干 chunk
   - → db.add_documents() → 向量化并写入 Weaviate
   - → db.as_retriever() → 标准检索器
   - → retriever.invoke(query) → 相关 Document 列表

8. 典型输出示例与观察点
   - 查询 "关于配置接口的信息有哪些"，配置 k=10 + score_threshold=0.5，
     得到 10 条文档的前 50 字符，例如：
       '接口说明：用于更新对应应用的调试长记忆内容，如果应用没有开启长记忆功能…'
       '如果接口需要授权，需要在 headers 中添加 Authorization ，并附加 access…'
       '1.2 [todo]更新应用草稿配置信息…'
       …（共 10 条）
     len(documents) == 10
   - 观察要点：这份 API 文档语料规模足够、且与查询高度同主题，
     所以 10 条都跨过了 0.5 阈值；换成小语料或跨主题查询时，
     返回条数会明显小于 k，这才是阈值真正在起作用的信号

9. 版本对照说明（示例基于 LangChain 1.x）
   - as_retriever() 的 search_type / search_kwargs 两个参数在 0.x → 1.x 之间未变
   - BaseRetriever.get_relevant_documents() 从 0.3.0 起被弃用，统一改用 Runnable 的 invoke()；
     本示例已按 1.x 写法使用 invoke()
   - 文档中 weaviate.connect_to_wcs() 在较新 weaviate 客户端里已更名为
     connect_to_weaviate_cloud()，老名字仍可用但会告警；迁移时需留意

10. 最佳实践建议
   - 业务代码始终面向 Retriever 编程，不要直接调用 VectorStore 的检索方法，便于后续替换实现
   - 使用 similarity_score_threshold 时务必同时传 k 与 score_threshold，缺一会告警或不生效
   - add_documents 幂等性较弱，重复执行脚本会导致库中出现重复数据；
     正式项目应先按业务主键去重，或用 delete + upsert 逻辑
   - API Key 硬编码在源码中仅适用于教学示例，生产环境必须通过 .env / 密钥管理服务注入
   - Weaviate 客户端是长连接资源，脚本型任务建议用 with 语句或显式 client.close() 释放
   - k 值过大会让 Prompt 超长并引入噪声，通常 3~10 条足够；阈值与 k 要配合调优


In [ ]:
import dotenv
import weaviate
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_weaviate import WeaviateVectorStore
from weaviate.auth import AuthApiKey
dotenv.load_dotenv()
loader = UnstructuredMarkdownLoader("./项目API文档.md")
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", "。|！|？", r"\.\s|\!\s|\?\s", r"；|;\s", r"，|,\s", " ", "", ],
    is_separator_regex=True,
    chunk_size=500,
    chunk_overlap=50,
    add_start_index=True,
)
documents = loader.load()
chunks = text_splitter.split_documents(documents)
db = WeaviateVectorStore(
    client=weaviate.connect_to_wcs(
        cluster_url="https://eftofnujtxqcsa0sn272jw.c0.us-west3.gcp.weaviate.cloud",
        auth_credentials=AuthApiKey("21pzYy0orl2dxH9xCoZG1O2b0euDeKJNEbB0"),
    ),
    index_name="DatasetDemo",
    text_key="text",
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
)
db.add_documents(chunks)
retriever = db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 10, "score_threshold": 0.5},
)
documents = retriever.invoke("关于配置接口的信息有哪些")
print(list(document.page_content[:50] for document in documents))
print(len(documents))